# Fitbit session-identifier revision audit

## tl;dr

- The case materials describe **30 consenting users**; the prepared files contain **35 session/export identifiers**, and their person-level mapping is not verifiable.
- No exact cross-session activity export block, sleep log, or weight record was found.
- The paired sleep comparison retains **19 eligible session identifiers** and estimates **0.70 more recorded hours on weekends** on average; this remains selective and non-causal.
- Removing correlated clustering inputs changes the original grouping materially (ARI 0.08); the revised two-group solution contains a three-session group and is not suitable as an operational taxonomy.
- Heart rate is validated at unique session-timestamp grain and prepared for a bounded non-medical appendix, but selective coverage prevents main or cross-session physiological claims.

## Context & Methods

This companion notebook reruns local calculations from validated BigQuery extracts and the saved read-only identifier audit outputs. It does not infer identity or merge session identifiers. The approved heart-rate transformation is documented separately in `reports/analysis/heart_rate_cleaning_qa.md`; the reproducible SQL lives under `sql/clean/`, `sql/qa/`, and `sql/analysis/`.

### Key Assumptions

- `profile_id` is a source field retained for lineage; reader-facing interpretation is session-level.
- Complete activity days contain 1,440 minute rows.
- Weekend sleep eligibility requires at least five weekday, three weekend, and ten total recorded sleep days.
- Session-bootstrap and leave-one-session-out checks cannot resolve unknown clustering of multiple sessions within one person.

## Data

In [1]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if not (ROOT / 'reports' / 'analysis').exists():
    raise RuntimeError(f'Run from the repository root or notebooks directory; resolved {ROOT}')
DATA = ROOT / 'reports' / 'analysis' / 'data'
inputs = {
    'daily_panel': DATA / 'daily_panel.csv',
    'identifier_coverage': DATA / 'audit_21_identifier_coverage.csv',
    'duplicate_activity_pairs': DATA / 'audit_24_duplicate_activity_pairs.csv',
    'heart_rate_clean_readiness': DATA / 'audit_33_heart_rate_clean_readiness.csv',
    'heart_rate_daily_metrics': DATA / 'heart_rate_daily_metrics.csv',
    'weight_readiness': DATA / 'audit_28_weight_bodyfat_readiness.csv',
}
pd.DataFrame([{'input': name, 'rows': len(pd.read_csv(path)), 'path': str(path.relative_to(ROOT))} for name, path in inputs.items()])

,input,rows,path
0,daily_panel,1935,reports/analysis/data/daily_panel.csv
1,identifier_coverage,19,reports/analysis/data/audit_21_identifier_cove...
2,duplicate_activity_pairs,592,reports/analysis/data/audit_24_duplicate_activ...
3,heart_rate_clean_readiness,15,reports/analysis/data/audit_33_heart_rate_clea...
4,heart_rate_daily_metrics,469,reports/analysis/data/heart_rate_daily_metrics...
5,weight_readiness,13,reports/analysis/data/audit_28_weight_bodyfat_...


## Results

In [2]:
import runpy
runpy.run_path(str(ROOT / 'scripts' / 'revise_identifier_analysis.py'), run_name='__main__')
stats = json.loads((ROOT / 'reports' / 'analysis' / 'revision_statistics.json').read_text())
{
    'case_study_consenting_users': stats['identifier_audit']['case_study_consenting_users'],
    'observed_session_identifiers': stats['identifier_audit']['observed_session_identifiers'],
    'underlying_people': stats['identifier_audit']['underlying_people_in_usable_files'],
    'confirmed_cross_session_duplicate_blocks': stats['duplicate_export_audit']['exact_full_sequence_days'],
}

{
  "session_identifiers": 35,
  "confirmed_cross_session_duplicate_blocks": 0,
  "paired_sleep_sessions": 19,
  "heart_rate_sessions": 15,
  "weight_sessions": 13,
  "clustering_reduced_silhouette": 0.44739630258264335
}


{'case_study_consenting_users': 30,
 'observed_session_identifiers': 35,
 'underlying_people': 'not verifiable from the identifier',
 'confirmed_cross_session_duplicate_blocks': 0}

### Identifier-weighting and complete-day sensitivity

In [3]:
mean_steps = stats['identifier_sensitivity']['mean_steps']
pd.DataFrame([
    {'estimate': 'Pooled session-day mean', 'steps': mean_steps['pooled_session_day']},
    {'estimate': 'Equal session-identifier mean', 'steps': mean_steps['equal_session_identifier']},
    {'estimate': 'Equal calendar-date mean', 'steps': mean_steps['equal_calendar_date']},
    {'estimate': 'Complete-day-only mean', 'steps': mean_steps['complete_day_only']},
]).round(1)

,estimate,steps
0,Pooled session-day mean,7200.4
1,Equal session-identifier mean,6856.9
2,Equal calendar-date mean,7195.7
3,Complete-day-only mean,7279.8


### Feature inclusion and exclusion

In [4]:
feature_matrix = pd.read_csv(ROOT / 'reports' / 'analysis' / 'feature_inclusion_exclusion_matrix.csv')
feature_matrix[['feature', 'session_identifiers_covered', 'usable_observations', 'observation_unit', 'decision', 'analytical_suitability']]

,feature,session_identifiers_covered,usable_observations,observation_unit,decision,analytical_suitability
0,Steps,35,1935,observed session-days,primary analysis,high for exploratory session-level analysis
1,Calories,35,1935,observed session-days,secondary analysis,strong within-session; limited across sessions
2,Activity intensity,35,1935,observed session-days,primary analysis,high for descriptive composition
3,Sedentary time,35,1935,observed session-days,primary analysis,moderate with sensitivity analysis
4,Sleep,25,832,sleep-covered session-days,secondary analysis,"secondary, paired or within-session only"
5,METs,35,1935,observed session-days,secondary analysis,supporting relative-intensity measure
6,Heart rate,15,469,daily heart-rate aggregate rows,appendix,bounded non-medical within-session exploration
7,Weight,13,98,unique session/log records,feature presence only,feature presence and cadence only
8,Body fat,3,4,non-null body-fat records,excluded,unusable
9,Distance,35,0,analysis-ready rows,excluded,not fit for cross-period primary analysis


### Segmentation stability

In [5]:
cluster_summary = stats['clustering']
pd.Series({
    'original_silhouette': cluster_summary['original_silhouette'],
    'reduced_silhouette': cluster_summary['reduced_silhouette'],
    'reduced_plus_METs_silhouette': cluster_summary['reduced_plus_mets_silhouette'],
    'original_vs_reduced_ARI': cluster_summary['original_vs_reduced_adjusted_rand_index'],
    'minimum_leave_one_session_out_ARI': cluster_summary['reduced_leave_one_session_out_min_adjusted_rand_index'],
    'minimum_reduced_group_size': cluster_summary['k_evaluation'][0]['minimum_group_size'],
}).round(3)

original_silhouette                  0.314
reduced_silhouette                   0.447
reduced_plus_METs_silhouette         0.238
original_vs_reduced_ARI              0.084
minimum_leave_one_session_out_ARI    0.034
minimum_reduced_group_size           3.000
dtype: float64

## Takeaways

1. Retain the validated daily and hourly base; revise the interpretation from people to session profiles.
2. Treat METs as a supporting relative-intensity measure, not a new marketing segment or independent outcome.
3. Keep cleaned heart rate appendix-only: use non-medical within-session metrics with coverage eligibility; prohibit resting-rate, zone, diagnostic, targeting, and population claims.
4. Limit weight to feature-data presence and unique-record cadence (100 staged rows, 98 unique session/log records); body fat and distance remain excluded.
5. Replace fixed activity clusters with continuous personal-baseline rules because feature selection and single-session omissions can change assignments materially.